# Phase 4 on Colab — closure: the audits, the repairs, the one table, the write-up

Sequencer for [`PHASE4_PLAN.md`](../PHASE4_PLAN.md). Phase 4 is **closure, not a new
experiment phase**: SCOPE Part 7's stop condition is met, and its own instruction is
"stop running experiments and start writing". The cells below order exactly that:
score the two registered audit gates, repair the nine known score-sheet mismatches,
assemble the one combined table, finish `run_all.sh`, and sync everything.

| Step | Command | Writes | Gate |
|---|---|---|---|
| audits (§3) | `python -m scripts.audit_sample make/score --conditions C5/C6` | `artifacts/audit_c5..6/` + report sections | blind; sheet committed before unblinding; §9.5 decision rules |
| repairs (§4) | `python -m embeded.evaluate --condition <c> --seed <s> --force` | re-graded `eval_metrics.json` | acceptance: zero npz↔metrics mismatches board-wide |
| one table (§5) | this notebook's builder cell | `## Phase 4 — the one table` | D4-1..D4-4 recorded first |
| run_all (§6) | `scripts/run_all.sh` rewrite + `bash -n` | the reproducing script | every report number has a command |
| sync (§8) | `python -m scripts.hf_artifacts push --include-report` | canonical report on the hub | writers only; readers never push |

Standing rules carried in: **no C5/C6 headline before its audit is scored**; limitations
written before claims; amendments appended, never edited; `python -u` children; a run's
settings fingerprint is its identity.

In [ ]:
# --- section 0: get the repo + the right branch into THIS kernel -------------
# (a fresh Colab kernel starts in /content with no repo on sys.path - every later
# cell depends on this one having run; if imports ever fail, re-run from here.)
import os, subprocess, sys
ROOT = '/content/EmbedEd'
BRANCH = 'arena/01a0e714-embeded'
MARKER = '5893b0f'   # the commit this notebook file was last edited at.
# Warns ONLY if the clone's tip does not contain this commit (i.e. this notebook is
# NEWER than the clone / diverged). The repo simply moving AHEAD is normal and silent:
# a prefix-match against HEAD could never stop warning, since a marker can never equal
# the commit that introduces it.

def _git(*args):
    r = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f'git {" ".join(args)} failed:\n{r.stderr}')
    return r.stdout.strip()

if not os.path.isdir(ROOT):
    print('cloning the repo...')
    _r = subprocess.run(['git', 'clone',
                         'https://github.com/ksu-gitreaper807/EmbedEd.git', ROOT],
                        capture_output=True, text=True)
    if _r.returncode:
        raise SystemExit('clone failed:\n' + _r.stderr)
os.chdir(ROOT)                                    # kernel-wide: later cells inherit this
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
_git('fetch', 'origin', BRANCH)
try:
    _git('checkout', '-B', BRANCH, f'origin/{BRANCH}')   # local branch tracks origin; VM clones hold no unique work
except SystemExit:
    _git('checkout', BRANCH)
head = _git('log', '--oneline', '-1')
print('repo ready:', head)
_anc = subprocess.run(['git', 'merge-base', '--is-ancestor', MARKER, 'HEAD'],
                      cwd=ROOT, capture_output=True)
if _anc.returncode != 0:
    print(f'[warn] this notebook was edited at {MARKER}, which is NOT in the clone\'s '
          f'history ({head.split()[0]}): the notebook is newer than the repo or the '
          'history diverged. If a later cell fails oddly, re-run THIS cell; if it still '
          'warns, download the tip\'s notebook and work from that copy.')

In [ ]:
# --- env: pins, secrets, settings (run after section 0) ----------------------
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 scikit-learn==1.6.1 pytest==8.3.2 matplotlib==3.9.4

import os, sys
ROOT = '/content/EmbedEd'
if not os.path.isdir(ROOT):
    raise SystemExit('the repo is not at /content/EmbedEd - run section 0 first (it '
                     'clones and checks out the right branch), then re-run this cell.')
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ.setdefault('EMBEDED_ARTIFACTS', '/content/embeded-artifacts')
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.makedirs(os.environ['EMBEDED_ARTIFACTS'], exist_ok=True)
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION',
                'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass

# identity probe BEFORE any multi-file call: anonymous bursts are how 429s happen
if os.environ.get('EMBEDED_HF_REPO_ID'):
    from huggingface_hub import HfApi
    try:
        _info = HfApi(token=os.environ.get('HF_TOKEN')).whoami()
        print(f"[hf] identity: {_info.get('name')} (token attached)")
    except Exception as exc:
        _status = getattr(getattr(exc, 'response', None), 'status_code', None)
        if _status == 429:
            raise SystemExit('already rate limited: wait ~10-15 min WITHOUT re-running.')
        if _status in (401, 403):
            raise SystemExit('HF_TOKEN rejected (401/403): fix the secret / notebook access.')
        print(f'[hf] identity probe failed ({type(exc).__name__}) - continuing')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
from embeded import settings as S
print('settings VERSION =', S.VERSION)

In [ ]:
# --- helpers -----------------------------------------------------------------
import json, subprocess, sys, time
from pathlib import Path

ROOT = '/content/EmbedEd'; ART = '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')

def sh(*args, allow_fail=False):
    # -u: stream child output live (block-buffered stdout once hid whole stages)
    print('$ python -m ' + ' '.join(args), flush=True)
    rc = subprocess.run([sys.executable, '-u', '-m', *args]).returncode
    if rc and not allow_fail:
        raise SystemExit(f'`python -m {args[0]}` exited {rc} - the reason is printed above.')
    return rc

def need(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing: raise SystemExit('missing file(s):\n  ' + '\n  '.join(missing))

def hf_main(argv):
    from scripts.hf_artifacts import main as m
    return m(argv)

def hf_pull(): return hf_main(['pull', '--if-configured', '--light', '--section',
                               'mining,runs-meta,sprime,report',
                               '--include', 'artifacts/audit*/**'])
def hf_push(*extra): return hf_main(['push', '--if-configured', *extra])

REPORT = Path(os.environ['EMBEDED_REPORT'])

def replace_or_append(section_header: str, block: str):
    text = REPORT.read_text() if REPORT.exists() else '# Measurements\n'
    import re
    if section_header in text:
        text = re.sub(re.escape(section_header) + r'.*?(?=\n## |\Z)',
                      lambda _m: block, text, flags=re.S)
    else:
        text = text.rstrip('\n') + '\n\n' + block
    REPORT.write_text(text)

# npz <-> stored-metrics mismatch detector (the §9.2 instrument; tolerance = rounding)
def audit_mismatches():
    import numpy as np
    root = S.ARTIFACTS / S.RUNS_SUBDIR
    out = []
    if not root.exists(): return out
    for p in sorted(root.glob('C[0-6]_1[345]/predictions.npz')):
        m = p.parent / 'eval_metrics.json'
        if not m.exists(): continue
        z = np.load(p)
        thr = float(np.asarray(z['threshold']).ravel()[0])
        y = np.asarray(z['test_labels']).astype(int); s = np.asarray(z['test_scores'])
        pred = (s >= thr).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum())
        fp = int(((pred == 1) & (y == 0)).sum())
        fn = int(((pred == 0) & (y == 1)).sum())
        f1 = 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)
        stored = json.loads(m.read_text()).get('test_f1')
        if stored is not None and abs(stored - f1) > 5e-4:
            out.append((p.parent.name, stored, round(f1, 4), round(stored - f1, 4)))
    return out

# fast sectioned pull (identity probe already ran in the env cell)
hf_pull()
print('helpers ready - VERSION', S.VERSION)

## 2. Entry gate — closure may start only from the shipped state

Checks: the Phase-3 table and the consolidation are in the report, the offline suite is
green, and the two audit gates are still open (they are §3's job). If any check fails,
fix that first — closure work on top of a broken base is worthless.

In [ ]:
need(str(REPORT))
text = REPORT.read_text()
for marker in ('## Phase 2 — main results', '## Phase 3 — generalisation',
               '## Gate G1s', '## Phase 3 — decisions'):
    if marker not in text:
        raise SystemExit(f'{marker!r} not in the report: the consolidation did not ship '
                         '(or the pull missed it) - reconcile that first.')
if '## Phase 4 — the one table' in text:
    print('[entry] the one table already exists - this is a replay; cells below are '
          'idempotent and will rebuild it.')
rc = sh('pytest', '-q', 'embeded/tests', allow_fail=True)
if rc:
    raise SystemExit(f'the offline suite failed (exit {rc}) - no closure work on a red base.')
print('entry gate OK: consolidated report present, suite green.')

## 3. The audit gates — `audit_c5` and `audit_c6` (the last mandatory gate)

Protocol unchanged since Phase 2: `make` samples 50 mined negatives per condition,
**blind** (no condition shown), the sheet is committed before anyone unblinds; a human
labels every pair; `score` verifies the sheet against the *current* triples (a stale
sheet is refused) and writes the rate with a Wilson interval to the report.

**Pre-registered predictions and decision rules (sanitycheck §9.5, recorded 2026-10-02
— before any C5/C6 run):**
- **audit_c5:** prediction 0–2% contamination. In band → C5 is a *confirmed control*
  and C5 sentences become claimable beside C1's.
- **audit_c6:** H1 predicts 25–35%. In band → H1 survives; §9.4's dose/similarity story
  extends to the semantic supply. **Far below band → H2: §9.4 and SCOPE §6's framing are
  rewritten** — the similarity-amplifier account was BM25-specific.

Endpoint order is fixed: scorer state → F1 → audited rate. A reader VM can do the
labelling; **only a writer VM runs the push at the end.**

In [ ]:
# --- 3a. make the two blind sheets (idempotent; never overwrites labelled work)
import csv, os
art = S.ARTIFACTS

def n_filled(labels_csv):
    # count NON-EMPTY labels: make pre-writes the csv with 50 id rows and an
    # empty label column, so counting lines would always read 50/50 on day zero.
    if not labels_csv.exists():
        return 0
    with open(labels_csv, encoding='utf-8') as fh:
        return sum(1 for r in csv.DictReader(fh) if (r.get('label') or '').strip())

for cond in ('C5', 'C6'):
    d = art / f'audit_{cond.lower()}'
    os.environ['EMBEDED_AUDIT_DIR'] = str(d)   # per-condition sheet; never the shared (and scored) Phase-2 audit/
    print(f'===== audit sheet: {cond} =====')
    rc = sh('scripts.audit_sample', 'make', '--conditions', cond,
            '--n', '50', '--seed', '13', allow_fail=True)
    if rc:
        print(f'(make exited {rc} for {cond} - usually: the sheet already exists and has '
              'labels, which make refuses to overwrite. That is correct; keep going.)')
os.environ.pop('EMBEDED_AUDIT_DIR', None)

for cond in ('C5', 'C6'):
    d = art / f'audit_{cond.lower()}'
    sheet, labels = d / 'audit_pairs.md', d / 'audit_labels.csv'
    if not sheet.exists():
        raise SystemExit(f'{sheet} missing - mining artifacts (triples_{cond}.jsonl) must '
                         'be pulled first (section 1).')
    n = n_filled(labels)
    print(f'{d.name}: sheet OK, labels filled: {n}/50')
    if n < 50:
        print(f'  -> HUMAN STEP: open {sheet}, judge each pair - do A and B implement the '
              'SAME functionality? clone / not_clone / unsure (unsure if it takes >1 min) - '
              f'and fill the label column of {labels}, save, then re-run this cell.')

missing = [c for c in ('C5', 'C6')
           if n_filled(art / f'audit_{c.lower()}' / 'audit_labels.csv') < 50]
if missing:
    raise SystemExit(f'audit label(s) incomplete for {missing}: fill the label column of '
                     'artifacts/audit_c5/audit_labels.csv and artifacts/audit_c6/audit_labels.csv '
                     '(clone | not_clone | unsure), then re-run this cell (it will pass to 3b).')
print('both sheets labelled - continue to 3b (scoring).')

In [ ]:
# --- 3b. score both sheets blind (each is verified against the current triples)
import json, os
rates = {}
for cond in ('C5', 'C6'):
    d = S.ARTIFACTS / f'audit_{cond.lower()}'
    os.environ['EMBEDED_AUDIT_DIR'] = str(d)   # score THIS condition's sheet, not audit/
    print(f'===== scoring: {cond} =====')
    rc = sh('scripts.audit_sample', 'score', allow_fail=True)
    if rc:
        raise SystemExit(f'scoring {cond} exited {rc} - usually a stale sheet vs re-mined '
                         'triples, or an unlabelled/mislabelled row; the output above says which.')
    m = json.loads((d / 'audit_result.json').read_text())[cond]
    rates[cond] = m
    print(f"{cond}: FN rate {m['fn_rate']:.1%} (Wilson 95% [{m['fn_rate_ci95'][0]:.1%}, "
          f"{m['fn_rate_ci95'][1]:.1%}]), unsure {m['unsure']}/50, upper incl. unsure "
          f"{m['fn_rate_upper_incl_unsure']:.1%}")
os.environ.pop('EMBEDED_AUDIT_DIR', None)

rules = chr(10).join([
    '',
    '--- 9.5 decision rules (pre-registered 2026-10-02; apply verbatim) ---',
    'audit_c5: rate in 0-2%  -> C5 CONFIRMED CONTROL; C5 becomes claimable beside C1.',
    'audit_c6: rate in 25-35% -> H1 SURVIVES; the 9.4 result extends to the semantic supply.',
    '          rate far below -> H2: the 9.4 + SCOPE section 6 framing are REWRITTEN',
    '          (the similarity-amplifier account was BM25-specific).',
    'Record the branch taken in sanitycheck 9.5 (append; never edit) BEFORE any',
    'headline sentence uses C5/C6.'])
print(rules)
hf_push()   # writer VMs only: sync the audit artifacts + report sections

## 4. The nine repairs — stored score sheets vs raw predictions (§9.2)

Nine C2/C3 benchmark runs have `eval_metrics.json` values that disagree with what their
own `predictions.npz` re-computes to (up to +0.0273). The raw prediction files are
authoritative (the board and the audits read them), so the repair is: **re-grade each
run from its checkpoint** (`evaluate --force` — era-tolerant scoring loads the recorded
weights and keeps the fingerprint in the provenance note; the strict *resume* gate is
untouched), push, and then require **zero mismatches board-wide**. The pre-repair
deltas are stated in the report so the fix is visible, not silent.

In [ ]:
# --- 4a. detect (idempotent instrument; run any time)
bad = audit_mismatches()
print(f'{len(bad)} run(s) with stored-vs-npz mismatch (tolerance 5e-4):')
for name, stored, recomputed, delta in bad:
    print(f'  {name:<10} stored {stored:.4f}  npz {recomputed:.4f}  delta {delta:+.4f}')
if not bad:
    print('none - the board is consistent; §4 is a no-op (still safe to re-run 4b).')

In [ ]:
# --- 4b. re-grade the mismatched runs from their checkpoints (~minutes each, GPU)
import re
bad = audit_mismatches()
if not bad:
    print('nothing to repair.')
else:
    for name, _stored, _npz, _d in bad:
        m = re.match(r'C([0-6])_(\d+)$', name)
        assert m, name
        cond, seed = f'C{m.group(1)}', m.group(2)
        sh('embeded.evaluate', '--condition', cond, '--seed', seed, '--force')
    hf_push()
    print('re-graded + pushed:', [b[0] for b in bad])

In [ ]:
# --- 4c. acceptance: zero mismatches board-wide, then rebuild the main table
bad = audit_mismatches()
if bad:
    raise SystemExit(f'{len(bad)} mismatch(es) remain: {bad} - the acceptance criterion '
                     'is ZERO board-wide. Investigate before rebuilding any table.')
sh('embeded.evaluate', '--results-table')
print('ACCEPTANCE MET: zero stored-vs-npz mismatches; main table regenerated. '
      'Note the pre-repair deltas (all C2/C3, <= +0.0273, no condition boundary '
      'crossed) in the write-up so the repair is visible, not silent.')

## 5. The one table + the closure decisions (PHASE4_PLAN §3, §9)

SCOPE Part 7 asks for *one table* with both test surfaces and the measured FN rates.
The two surfaces use different rulers (13.7%-positive natural prevalence vs balanced
slices — the same base model scores 0.2572 and 0.6934), so the table carries an explicit
non-comparability note and the within-corpus anchor (the untuned model's s′ score).

**Record the four closure decisions first** (D4-1 layout, D4-2 headline condition,
D4-3 the ≈0.95 framing, D4-4 the extension gate) — recommendations pre-filled from
PHASE4_PLAN §9; edit `DECISIONS` only with a written reason.

In [ ]:
# --- 5a. record D4-1..D4-4 (before the table is built)
DECISIONS = {
    'D4-1': {'choice': 'three panels inline (benchmark / generalisation / FN rates) '
                       '+ non-comparability note; C4-C6 inline, flagged as the recorded '
                       'factorial extension',
             'rationale': 'Part 7 names C0-C3; the bouncer trio joined by the recorded '
                          '2026-10-03 addendum - visible, not smuggled in.'},
    'D4-2': {'choice': 'headline condition = C1; C5 only if audit_c5 lands in band',
             'rationale': "C1 is the best *trained* condition with its gate closed; C5's "
                          'audit is the open one (section 3).'},
    'D4-3': {'choice': 'the ~0.95 published number is framed as a pair-regime difference '
                       '(reconciliation as a limitation)',
             'rationale': 'Corpus-wide natural prevalence (ceiling 0.2406) vs the '
                          'near-clone-heavy published splits; MAP@R carries the '
                          'prevalence-robust companion. Not chased as a harness defect.'},
    'D4-4': {'choice': 'the denoising delta-ladder on C3, ONLY after a completed draft '
                       'AND a SCOPE amendment',
             'rationale': "SCOPE's own 'natural next experiment' (Part 6 rule 7 keeps the "
                          'bar: a clear 9-page write-up beats a half-finished extension).'},
}
path = S.ARTIFACTS / 'phase4_decisions.json'
stamp = time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())
payload = {'recorded_utc': stamp, 'version': S.VERSION, 'decisions': DECISIONS}
if path.exists() and json.loads(path.read_text()).get('decisions') == DECISIONS:
    print('[decisions] unchanged - replay, original record untouched:',
          json.loads(path.read_text())['recorded_utc'])
else:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    print(json.dumps(payload, indent=2, ensure_ascii=False))
    print(f'wrote {path}')

In [ ]:
# --- 5b. build '## Phase 4 — the one table' (regenerated, never hand-edited)
import numpy as np

def _runs(kind_prefix):
    root = S.ARTIFACTS / S.RUNS_SUBDIR
    rows = {}
    for p in sorted(root.glob(kind_prefix + '*/eval_metrics.json')):
        m = json.loads(p.read_text())
        rows.setdefault(m['condition'], []).append(m)
    return rows

bench = _runs('C[0-6]_1[345]')                      # seeds 13-15 = the registered ladder
spr = _runs('sprime_C[1-6]_1[345]')
c0_anchor = None
tp = S.ARTIFACTS / S.RUNS_SUBDIR / 'sprime_transfer_C0_13' / 'eval_metrics.json'
if tp.exists():
    j = json.loads(tp.read_text()); c0_anchor = (j['f1_seen'], j['f1_unseen'])

lines = ['## Phase 4 — the one table', '',
         f'Assembled from the run dirs by the Phase-4 notebook — version `{S.VERSION}`, '
         f'recorded {time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())}. '
         'Sources: `runs/<run>/eval_metrics.json`, the audit sections below, and the '
         's′ table. **The two panels are different rulers** (benchmark: natural 13.7% '
         'prevalence, all-negative ceiling 0.2406; s′: balanced slices) — absolute F1 '
         'is NOT comparable across panels; within a panel, orderings are the finding.', '',
         '### Panel A — benchmark (F1 mean ± sd over seeds 13-15; pinned = validation '
         'thresholds saturated at 1.0)', '',
         '| condition | F1 | pinned | audited/measured FN rate |', '|---|---|---|---|']
FN = {'C1': '≈0-2% (audited)', 'C2': '42% (audited)', 'C3': '30% [0.191, 0.438] (audited)',
      'C4': '30% [0.191, 0.438] (audited)', 'C5': '0.27% supply (audit_c5: §3)',
      'C6': '13.6% supply (audit_c6: §3)'}
for c in ('C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6'):
    ms = bench.get(c) or []
    if not ms: continue
    f1s = [m['test_f1'] for m in ms]
    pinned = sum(1 for m in ms if float(m.get('threshold', 0)) >= 0.999)
    mean = float(np.mean(f1s)); sd = float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0
    lines.append(f'| {c} | {mean:.4f} ± {sd:.4f} | {pinned}/{len(ms)} | {FN.get(c, "—")} |')
lines += ['', '*(Phase-2 extension seeds 16-18 exist on the hub and are reported as '
             'extra evaluations, not table rows.)*', '',
          '### Panel B — s′ generalisation (F1_seen / F1_unseen / Δ, seeds 13-15)', '',
          '| condition | F1_seen | F1_unseen | Δ |', '|---|---|---|---|']
for c in ('C1', 'C2', 'C3', 'C4', 'C5', 'C6'):
    ms = spr.get(c) or []
    if not ms: continue
    s_ = [m['f1_seen'] for m in ms]; u_ = [m['f1_unseen'] for m in ms]
    d_ = [a - b for a, b in zip(s_, u_)]
    lines.append(f'| {c} | {np.mean(s_):.4f} | {np.mean(u_):.4f} | {np.mean(d_):+.4f} |')
if c0_anchor:
    lines.append(f'| C0 (untuned anchor, transfer) | {c0_anchor[0]:.4f} | '
                 f'{c0_anchor[1]:.4f} | — |')
lines += ['', '**The floor:** no poisoned-mining condition beats the untuned encoder '
          'on unseen functionality; only clean training (C1, and C5 pending its audit '
          'branch) converts fine-tuning into gains.', '',
          '### Panel C — mechanism bookkeeping', '',
          'Trigger = an FN that is simultaneously a functional clone and a J ≥ 0.40 '
          'near-duplicate; rate = dose; margin 0.10 = amplifier (§9.4 FINAL). The '
          '40% cut removes ~40% of C2 contamination and 0 audited clean pairs. '
          'audit_c4 PASSED (30% ≤ 42%); audit_c5/c6 outcomes: see section 3 — no C5/C6 '
          'headline before its branch is recorded.']
replace_or_append('## Phase 4 — the one table', '\n'.join(lines) + '\n')
print('\n'.join(lines[:24]))
print('...')
print(f'wrote "## Phase 4 — the one table" in {REPORT}')

## 6. `run_all.sh` — every number gets one command (SCOPE item 22)

The script still says the Phase-3 deliverables are 'remaining' — they shipped. This cell
rewrites it as the complete, ordered pipeline (each stage artifact-gated and idempotent,
like the notebooks), syntax-checks it, and commits it.

In [ ]:
# --- 6a. rewrite scripts/run_all.sh (complete pipeline) + bash -n + commit
script = '''#!/usr/bin/env bash
# Reproduces every number in report/measurements.md (all phases). On Colab run
# the notebooks instead - same commands, same order, with per-cell gates.
set -euo pipefail
cd "$(dirname "$0")/.."
PY="${PYTHON:-python3}"

# --- Phase 0: corpus verify + s' acquisition ---------------------------------
$PY -m embeded.data.prepare_data --hf --verify-spec          # gate G0
$PY -m scripts.fetch_sprime                                  # s' gate (zenodo)

# --- Phase 1: embeddings, mining, tests, gates --------------------------------
$PY -m embeded.mining.semantic_index                          # GPU
$PY -m embeded.negatives --strategies random,bm25,semantic,filtered_random,filtered_bm25,filtered_semantic
$PY -m pytest -q embeded/tests
$PY -m scripts.hardness_diagnostics                           # no verdict; scale + floors
$PY -m embeded.hardcheck                                      # gate G1 (rule D1)

# --- Phase 2: benchmark campaign (C0 + C1-C6, seeds 13-15) --------------------
$PY -m embeded.train --condition C1 --seed 13 --smoke         # harness check (~3 min)
for c in C1 C2 C3 C4 C5 C6; do for s in 13 14 15; do
  $PY -m embeded.train    --condition "$c" --seed "$s"
  $PY -m embeded.evaluate --condition "$c" --seed "$s"
done; done
$PY -m embeded.evaluate --condition C0                        # untuned, evaluated once
$PY -m embeded.evaluate --results-table                       # main table

# --- Phase 3: s' generalisation + figure + demo (PHASE3_PLAN) -----------------
$PY -m embeded.generalize --census --holdout-k 3              # D4 rule, before any F1
$PY -m embeded.generalize --mine --holdout-k 3 --k 20 --cap 16000
$PY -m embeded.generalize --hardness --holdout-k 3            # gate G1s
$PY -m embeded.generalize --condition C1 --seed 13 --smoke --holdout-k 3
for c in C1 C2 C3 C4 C5 C6; do for s in 13 14 15; do
  $PY -m embeded.generalize --condition "$c" --seed "$s" --holdout-k 3
done; done
for c in C1 C2 C3 C4 C5 C6; do for s in 13 14 15; do
  $PY -m embeded.generalize --condition "$c" --seed "$s" --holdout-k 3 --transfer
done; done
$PY -m embeded.generalize --condition C0 --seed 13 --holdout-k 3 --transfer
$PY -m embeded.generalize --results-table
$PY -m embeded.visualize --panels C0,C1,C2,C3,C4,C5,C6
$PY -m demo.app --check

# --- optional sync (off by default: writers only) ------------------------------
# $PY -m scripts.hf_artifacts push --include-report
'''
Path('scripts/run_all.sh').write_text(script)
import subprocess
r = subprocess.run(['bash', '-n', 'scripts/run_all.sh'], capture_output=True, text=True)
if r.returncode:
    raise SystemExit('run_all.sh failed bash -n: ' + r.stderr)
print('run_all.sh rewritten + syntax OK (', len(script.splitlines()), 'lines )')
r = subprocess.run(['git', 'add', 'scripts/run_all.sh'], capture_output=True, text=True)
r = subprocess.run(['git', 'commit', '-q', '-m',
                    'run_all: the complete pipeline - every report number has one command '
                    '(SCOPE item 22; PHASE4 section 6)'], capture_output=True, text=True)
print('committed' if r.returncode == 0 else 'nothing to commit')

## 7. The write-up checklist (limitations BEFORE claims — SCOPE item 20)

Tick these in order; the draft is `paper/paper.tex` (kept pointer-accurate to the
snapshot, never a copy of it).

- [ ] **Limitations drafted first**: WT3/T4 label noise (93% figure; the mined-hard
      region is the dirtiest); the base-rate trap (0.2478 ≈ the 0.2406 ceiling — read
      with the ceiling and MAP@R); two-ruler non-comparability; one language/encoder/
      loss; three seeds, no p-values; G1/G1s as the manipulation checks; the §4 repairs
      and their pre-repair deltas (≤ +0.027, no boundary crossed); the ≈0.95 framing
      (D4-3: reconciliation, not chase); the bouncer's operating point is corpus-specific.
- [ ] **Claims ledger**: every headline sentence → (table panel, run dirs, audit gate
      status). C5/C6 sentences only after section 3's branch is recorded.
- [ ] **One table** (§5) inserted/linked from the report and the paper.
- [ ] `run_all.sh` (§6) referenced as the reproduction path.
- [ ] Fill the paper's two `[pending]` markers as §3/§4 land; swap the author block.
- [ ] Only then: D4-4's extension may be *proposed* — SCOPE amendment + completed draft
      first (Part 6 rule 7).

## 8. Final sync — the canonical report

One push that uploads the report (trainers never did; this is the consolidating VM), the
zip for your local archive, and the GitHub push. **Never from a reader VM.**

In [ ]:
hf_push('--include-report')
import shutil
shutil.make_archive('/content/phase4_report', 'zip', Path(ROOT) / 'report')
try:
    from google.colab import files
    files.download('/content/phase4_report.zip')
except Exception as exc:
    print(f'(download skipped: {exc.__class__.__name__}) - report/ is in the HF checkpoint '
          'and the VM clone.')
print('Phase 4 sync done: report (with audits, repairs, the one table) is canonical.')
print('Last human steps: push the branch to GitHub (the git-push cell), update '
      'PHASE4_PLAN section 10 with the execution-log row, then write.')